In [43]:
# ── Setup (run this cell first) ──────────────────────────────────────────────
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns
import re

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import xgboost as xgb

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
RANDOM_STATE = 42
DATA_DIR = r'/Users/pai/Desktop/MEAM/CIS 5450/steam_analysis/steam_dataset_2025_csv'
print('Setup complete.')

Setup complete.


Data cleanning procedures
First decide what games are we going to use for analysis
1. select paid games
2. select "game", not "dlc" or others
3. select games that don't discount. developers won't know whether to discount at deciding procedures, will lead to data leakage. (80854 items)
4. select games pricing with USD

Colums we need to keep 
appid, name, release_date, required_age, supported_languages (numbers), 'mat_supports_windows', 'mat_supports_mac', 'mat_supports_linux', 'mat_initial_price', 'mat_achievement_count' (convert to a binary variable indicating whether the game has a steam achievement system)

To make a main_df, we still need categories, genres, estimated sales, revenue, developers, publishers of the game.
Categories: First we decide what category to include in our main_df. In the original csv, we have hundreds of categories, and we choose the most common used 20 categories (cover 99.8% of our selected games). We include these categories in One-Hot Encoding form.

Genres: same as categories, we choose 20 most common seen genres.

Developers and publishers: we computed how many games a developer or publisher has made and include these in the main_df to indicate company's influence.

Sales and revenue: we basically calculate sales according to recommendation_totals, through a formula: recommendation_totals * 30.
For those have 0 in recommendation_totals, we calculate sales through computing how many reviews does a game has and implementing formula: sales = reviews * Boxleiter Multiplier (I chose 10) + 100, we get the sales. We get revenue by sales * price



In [44]:
# data cleaning 
apps       = pd.read_csv(f'{DATA_DIR}/applications.csv', low_memory=False)
genres_ref = pd.read_csv(f'{DATA_DIR}/genres.csv')
app_genres = pd.read_csv(f'{DATA_DIR}/application_genres.csv')
cats_ref   = pd.read_csv(f'{DATA_DIR}/categories.csv')
app_cats   = pd.read_csv(f'{DATA_DIR}/application_categories.csv')
devs_ref   = pd.read_csv(f'{DATA_DIR}/developers.csv')
app_devs   = pd.read_csv(f'{DATA_DIR}/application_developers.csv')
pubs_ref   = pd.read_csv(f'{DATA_DIR}/publishers.csv')
app_pubs   = pd.read_csv(f'{DATA_DIR}/application_publishers.csv')
reviews    = pd.read_csv(f'{DATA_DIR}/reviews.csv')

In [45]:
num = apps["recommendations_total"]
test = reviews[reviews["appid"]==1279450]
test


,recommendationid,appid,author_steamid,author_num_games_owned,author_num_reviews,author_playtime_forever,author_playtime_last_two_weeks,author_playtime_at_review,author_last_played,language,...,voted_up,votes_up,votes_funny,weighted_vote_score,comment_count,steam_purchase,received_for_free,written_during_early_access,created_at,updated_at
494172,202239479,1279450,76561198060312515,907,177,71.0,0.0,71.0,1.755338e+09,koreana,...,True,0,0,0.5,0,True,False,False,2025-09-07 12:51:00.564782+00:00,2025-09-09 02:00:35.549692+00:00


In [46]:
# filter out free games, non-game items, discounted games, non-USD games
apps_cleaned = apps.copy()
apps_cleaned = apps_cleaned[(apps_cleaned["is_free"]==False) & (apps_cleaned["mat_initial_price"]>0)]
apps_cleaned = apps_cleaned[apps_cleaned["type"]=="game"]
#apps_cleaned = apps_cleaned[apps_cleaned["mat_discount_percent"]==0]
apps_cleaned = apps_cleaned[apps_cleaned["mat_currency"]=="USD"]
apps_cleaned["mat_initial_price"] = apps_cleaned["mat_initial_price"] / 100.0

# filter the columns we need
apps_cleaned = (apps_cleaned[["appid", "name", "release_date", "required_age", 
    "supported_languages", "mat_supports_windows", "mat_supports_mac", "mat_supports_linux", 
    "mat_initial_price", "mat_achievement_count", "recommendations_total"]])
apps_cleaned.rename(columns={"mat_initial_price": "price_usd", 
    "mat_supports_windows": "supports_windows", "mat_supports_mac": "supports_mac", 
    "mat_supports_linux": "supports_linux"}, inplace=True)

# seperate the release_date column into release_year and release_month
apps_cleaned["release_date"] = pd.to_datetime(apps_cleaned["release_date"], errors='coerce')
apps_cleaned["release_year"] = apps_cleaned["release_date"].dt.year
apps_cleaned["release_month"] = apps_cleaned["release_date"].dt.month
apps_cleaned = apps_cleaned.dropna(subset=['release_year', 'release_month'])
apps_cleaned["release_year"]  = apps_cleaned["release_year"].astype(int)
apps_cleaned["release_month"] = apps_cleaned["release_month"].astype(int)

apps_cleaned.drop(columns=["release_date"], inplace=True)

# convert required_age to boolean: 0 = under 18 (all ages), 1 = 18+ (adult only)
apps_cleaned["required_age"] = pd.to_numeric(apps_cleaned["required_age"], errors='coerce').fillna(0)
apps_cleaned["required_age"] = (apps_cleaned["required_age"] >= 18).astype(int)

# parse supported_languages into a language count (int)
def _count_languages(s):
    if pd.isna(s):
        return 1                              
    s = str(s).split('<br>')[0]              
    s = re.sub(r'<[^>]+>', '', s)            
    langs = [l.strip() for l in s.split(',') if l.strip()]
    return max(len(langs), 1)

apps_cleaned["supported_languages"] = apps_cleaned["supported_languages"].apply(_count_languages).astype(int)
apps_cleaned["supported_languages"].value_counts()

# convert mat_achievement_count to boolean "achievement_systems"
apps_cleaned.rename(columns={"mat_achievement_count": "achievement_systems"}, inplace=True)
apps_cleaned["achievement_systems"] = apps_cleaned["achievement_systems"].notna().astype(int)

apps_cleaned = apps_cleaned.reset_index(drop=True)

In [47]:
# Filter out Categories
# ── Filter app_cats to games in apps_cleaned → app_cats_1 ────────────────────
valid_appids = set(apps_cleaned["appid"])
total_games = len(valid_appids)
app_cats_1 = app_cats[app_cats["appid"].isin(valid_appids)].copy()

# Merge with cats_ref to attach category names
cats_ref_eng = cats_ref.rename(columns={"id": "category_id", "name": "category_name"})
app_cats_1 = app_cats_1.merge(cats_ref_eng, on="category_id", how="left")

# Count distinct games per category, sort descending → pick top 20
cat_game_count = (
    app_cats_1.groupby(["category_id", "category_name"])["appid"]
    .nunique()
    .reset_index()
    .rename(columns={"appid": "game_count"})
    .sort_values("game_count", ascending=False)
    .reset_index(drop=True)
)
cat_top20 = cat_game_count.head(20)

# Filter out Genres
app_genres_1 = app_genres[app_genres["appid"].isin(valid_appids)].copy()

# Merge with genres_ref to attach genre names
genres_ref_eng = genres_ref.rename(columns={"id": "genre_id", "name": "genre_name"})
app_genres_1 = app_genres_1.merge(genres_ref_eng, on="genre_id", how="left")

# Count distinct games per genre, sort descending → pick top 20
genre_game_count = (
    app_genres_1.groupby(["genre_id", "genre_name"])["appid"]
    .nunique()
    .reset_index()
    .rename(columns={"appid": "game_count"})
    .sort_values("game_count", ascending=False)
    .reset_index(drop=True)
)
genre_top20 = genre_game_count.head(20) 

In [48]:
# proving the top 20 categories cover over 90% of the games
# Build lookup: category_id -> set of appids
cat_to_games = (
    app_cats_1.groupby("category_id")["appid"]
    .apply(set)
    .to_dict()
)

# Compute unique games covered by top-20 categories (union, no double-counting)
covered = set()
for cid in cat_top20["category_id"]:
    covered |= cat_to_games.get(cid, set())
coverage_pct = len(covered) / total_games * 100

# ── Output ────────────────────────────────────────────────────────────────────
print(f"Total games in apps_cleaned : {total_games:,}")
print(f"Games covered by top-20 cats: {len(covered):,}  ({coverage_pct:.1f}%)")
print(f"Threshold (90%)             : {int(total_games * 0.9):,}")
print(f"Passes 90% threshold        : {'YES ✓' if coverage_pct >= 90 else 'NO ✗'}\n")

print(f"{'#':<4} {'Category Name':<42} {'Games':>8}  {'Individual%':>11}  {'Cumulative%':>12}")
print("─" * 82)
cum_covered = set()
for i, row in cat_top20.iterrows():
    cum_covered |= cat_to_games.get(row["category_id"], set())
    print(f"{i+1:<4} {row['category_name']:<42} {row['game_count']:>8,}  "
          f"{row['game_count']/total_games*100:>10.1f}%  "
          f"{len(cum_covered)/total_games*100:>11.1f}%")

top20_cat_names = cat_top20["category_name"].tolist()   
print(f"\nTop-20 category names:\n{top20_cat_names}")

Total games in apps_cleaned : 88,917
Games covered by top-20 cats: 88,167  (99.2%)
Threshold (90%)             : 80,025
Passes 90% threshold        : YES ✓

#    Category Name                                 Games  Individual%   Cumulative%
──────────────────────────────────────────────────────────────────────────────────
1    Family Sharing                               87,537        98.4%         98.4%
2    Single-player                                85,689        96.4%         98.8%
3    Steam Achievements                           45,313        51.0%         98.9%
4    Steam Cloud                                  25,011        28.1%         98.9%
5    Full controller support                      20,471        23.0%         98.9%
6    Multi-player                                 14,214        16.0%         99.0%
7    Partial Controller Support                   10,998        12.4%         99.0%
8    Steam Trading Cards                          10,073        11.3%         99.0%
9   

In [49]:
# proving the top 20 genres cover over 90% of the games
# Build lookup: genre_id -> set of appids
genre_to_games = (
    app_genres_1.groupby("genre_id")["appid"]
    .apply(set)
    .to_dict()
)

# Compute unique games covered by top-20 genres (union, no double-counting)
covered = set()
for gid in genre_top20["genre_id"]:
    covered |= genre_to_games.get(gid, set())
coverage_pct = len(covered) / total_games * 100

# ── Output ────────────────────────────────────────────────────────────────────
print(f"Total games in apps_cleaned   : {total_games:,}")
print(f"Games covered by top-20 genres: {len(covered):,}  ({coverage_pct:.1f}%)")
print(f"Threshold (90%)               : {int(total_games * 0.9):,}")
print(f"Passes 90% threshold          : {'YES ✓' if coverage_pct >= 90 else 'NO ✗'}\n")

print(f"{'#':<4} {'Genre Name':<42} {'Games':>8}  {'Individual%':>11}  {'Cumulative%':>12}")
print("─" * 82)
cum_covered = set()
for i, row in genre_top20.iterrows():
    cum_covered |= genre_to_games.get(row["genre_id"], set())
    print(f"{i+1:<4} {row['genre_name']:<42} {row['game_count']:>8,}  "
          f"{row['game_count']/total_games*100:>10.1f}%  "
          f"{len(cum_covered)/total_games*100:>11.1f}%")

top20_genre_names = genre_top20["genre_name"].tolist()
print(f"\nTop-20 genre names:\n{top20_genre_names}")


Total games in apps_cleaned   : 88,917
Games covered by top-20 genres: 88,769  (99.8%)
Threshold (90%)               : 80,025
Passes 90% threshold          : YES ✓

#    Genre Name                                    Games  Individual%   Cumulative%
──────────────────────────────────────────────────────────────────────────────────
1    Indie                                        63,613        71.5%         71.5%
2    Casual                                       39,372        44.3%         81.8%
3    Action                                       36,166        40.7%         89.3%
4    Adventure                                    35,883        40.4%         92.7%
5    Simulation                                   19,058        21.4%         95.3%
6    Strategy                                     17,603        19.8%         97.1%
7    RPG                                          16,219        18.2%         98.3%
8    Early Access                                  8,615         9.7%         98

In [50]:
# ── Developer & Publisher features ───────────────────────────────────────────
# dev_game_count / pub_game_count: count from ALL games in the DB (not just apps_cleaned)
devs_ref_m = devs_ref.rename(columns={"id": "developer_id", "name": "developer_name"})
pubs_ref_m = pubs_ref.rename(columns={"id": "publisher_id", "name": "publisher_name"})

dev_counts_all = (
    app_devs.groupby("developer_id")["appid"]
    .nunique().reset_index()
    .rename(columns={"appid": "dev_game_count"})
)
pub_counts_all = (
    app_pubs.groupby("publisher_id")["appid"]
    .nunique().reset_index()
    .rename(columns={"appid": "pub_game_count"})
)

# Filter to apps_cleaned, attach names + all-games counts
app_devs_1 = (
    app_devs[app_devs["appid"].isin(valid_appids)].copy()
    .merge(devs_ref_m, on="developer_id", how="left")
    .merge(dev_counts_all, on="developer_id", how="left")
)
app_pubs_1 = (
    app_pubs[app_pubs["appid"].isin(valid_appids)].copy()
    .merge(pubs_ref_m, on="publisher_id", how="left")
    .merge(pub_counts_all, on="publisher_id", how="left")
)

# Primary dev/pub per game (sort by id → take first)
primary_dev = (
    app_devs_1.sort_values("developer_id")
    .groupby("appid", as_index=False).first()
    [["appid", "developer_name", "dev_game_count"]]
)
primary_pub = (
    app_pubs_1.sort_values("publisher_id")
    .groupby("appid", as_index=False).first()
    [["appid", "publisher_name", "pub_game_count"]]
)

# is_self_published: 1 if any dev name ∩ any pub name for that game
dev_name_sets = app_devs_1.groupby("appid")["developer_name"].apply(set)
pub_name_sets  = app_pubs_1.groupby("appid")["publisher_name"].apply(set)
name_cmp = dev_name_sets.to_frame().join(pub_name_sets, how="outer")
name_cmp["is_self_published"] = name_cmp.apply(
    lambda r: int(bool(
        (r["developer_name"] if isinstance(r["developer_name"], set) else set()) &
        (r["publisher_name"]  if isinstance(r["publisher_name"],  set) else set())
    )), axis=1
)
is_self_pub = name_cmp[["is_self_published"]].reset_index()

# ── Calculate estimated sales ────────────────────────────────────────────────
# Primary: recommendations_total * 30  (if > 0)
# Fallback: review_count * 10 + 100   (if recommendations_total == 0 or NaN)

# Prepare reviews table: count per game for apps_cleaned only
reviews_slim = reviews[["recommendationid", "appid"]]
review_counts = (
    reviews_slim[reviews_slim["appid"].isin(valid_appids)]
    .groupby("appid")["recommendationid"]
    .count()
    .reset_index()
    .rename(columns={"recommendationid": "review_count"})
)

# Normalise recommendations_total to numeric
apps_cleaned["recommendations_total"] = pd.to_numeric(
    apps_cleaned["recommendations_total"], errors="coerce"
).fillna(0)

# ── Category OHE (top-20) ─────────────────────────────────────────────────────
def _clean_col(prefix, name):
    return prefix + re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")

top20_cat_ids = set(cat_top20["category_id"])
cat_ohe = app_cats_1[app_cats_1["category_id"].isin(top20_cat_ids)].copy()
cat_ohe["col"] = cat_ohe["category_name"].apply(lambda n: _clean_col("cat_", n))
cat_pivot = (
    cat_ohe.assign(val=1)
    .pivot_table(index="appid", columns="col", values="val", aggfunc="max", fill_value=0)
    .reset_index()
)

# ── Genre OHE (top-20) ────────────────────────────────────────────────────────
top20_genre_ids = set(genre_top20["genre_id"])
genre_ohe = app_genres_1[app_genres_1["genre_id"].isin(top20_genre_ids)].copy()
genre_ohe["col"] = genre_ohe["genre_name"].apply(lambda n: _clean_col("genre_", n))
genre_pivot = (
    genre_ohe.assign(val=1)
    .pivot_table(index="appid", columns="col", values="val", aggfunc="max", fill_value=0)
    .reset_index()
)

# ── Build main_df ─────────────────────────────────────────────────────────────
main_df = apps_cleaned.copy()
main_df = main_df.merge(primary_dev,                              on="appid", how="left")
main_df = main_df.merge(primary_pub[["appid","publisher_name","pub_game_count"]], on="appid", how="left")
main_df = main_df.merge(is_self_pub,                             on="appid", how="left")
main_df = main_df.merge(cat_pivot,                               on="appid", how="left")
main_df = main_df.merge(genre_pivot,                             on="appid", how="left")
main_df = main_df.merge(review_counts, on="appid", how="left")
main_df["review_count"] = main_df["review_count"].fillna(0).astype(int)

# Primary path: recommendations_total > 0  →  estimated_sales = recommendations_total * 30
# Fallback    : recommendations_total == 0 →  estimated_sales = review_count * 10 + 100
has_reco = main_df["recommendations_total"] > 0
main_df["estimated_sales"] = 0
main_df.loc[has_reco,  "estimated_sales"] = (main_df.loc[has_reco,  "recommendations_total"] * 30).astype(int)
main_df.loc[~has_reco, "estimated_sales"] = (main_df.loc[~has_reco, "review_count"] * 10 + 100).astype(int)
main_df["revenue"] = (main_df["price_usd"] * main_df["estimated_sales"]).round(2)

# Fill missing
for col in ["dev_game_count", "pub_game_count", "is_self_published"]:
    main_df[col] = main_df[col].fillna(0).astype(int)
cat_cols   = [c for c in main_df.columns if c.startswith("cat_")]
genre_cols = [c for c in main_df.columns if c.startswith("genre_")]
main_df[cat_cols + genre_cols] = main_df[cat_cols + genre_cols].fillna(0).astype(int)

# ── Summary ───────────────────────────────────────────────────────────────────
main_df.drop(columns=["recommendations_total", "review_count"], inplace=True)
main_df = main_df.sort_values(by="revenue", ascending=False)
main_df = main_df.reset_index(drop=True)

In [51]:
main_df.to_csv("main_df2.csv", index=False)
main_df[main_df["appid"]==504230]
main_df.head(20)


,appid,name,required_age,supported_languages,supports_windows,supports_mac,supports_linux,price_usd,achievement_systems,release_year,...,genre_racing,genre_rpg,genre_simulation,genre_sports,genre_strategy,genre_utilities,genre_video_production,genre_violent,estimated_sales,revenue
0,2358720,Black Myth: Wukong,0,14,True,True,True,59.99,1,2024,...,0,1,0,0,0,0,0,0,25659840,1.539334e+09
1,1091500,Cyberpunk 2077,0,19,True,True,True,59.99,1,2020,...,0,1,0,0,0,0,0,0,23601720,1.415867e+09
2,1245620,ELDEN RING,0,15,True,True,True,59.99,1,2022,...,0,1,0,0,0,0,0,0,23526240,1.411339e+09
3,252490,Rust,0,25,True,True,True,39.99,1,2018,...,0,1,0,0,0,0,0,0,31301130,1.251732e+09
4,1086940,Baldur's Gate 3,0,15,True,True,True,59.99,1,2023,...,0,1,0,0,1,0,0,0,20625990,1.237353e+09
5,292030,The Witcher 3: Wild Hunt,0,17,True,True,True,39.99,1,2015,...,0,1,0,0,0,0,0,0,23542050,9.414466e+08
6,553850,HELLDIVERS™ 2,0,14,True,True,True,39.99,1,2024,...,0,0,0,0,0,0,0,0,22796100,9.116160e+08
7,221100,DayZ,0,12,True,True,True,49.99,1,2018,...,0,0,0,0,0,0,0,0,10883250,5.440537e+08
8,275850,No Man's Sky,0,14,True,True,True,59.99,1,2016,...,0,0,0,0,0,0,0,0,8206050,4.922809e+08
9,374320,DARK SOULS™ III,0,12,True,True,True,59.99,1,2016,...,0,0,0,0,0,0,0,0,7967430,4.779661e+08
